# Module 22 – Naive Bayes Case Study on Titanic Dataset

**Name:** ____________________  
**Course/Batch:** ____________________  
**Date:** ____________________

## Objective

Build a Titanic survival prediction model using the **Naive Bayes** classification algorithm.

This notebook covers:
1. Dataset import and exploration
2. Data cleaning and preprocessing
3. Encoding categorical variables
4. Train/test splitting
5. Naive Bayes model training
6. Model evaluation
7. Confusion matrix and classification report
8. Short analysis and insights

> **Note:** This is an educational machine-learning exercise. It should not be treated as a historical or real-world predictive system beyond the supplied dataset.


## 1. Import Required Libraries

We use:
- **pandas** for data loading and manipulation
- **numpy** for numerical operations
- **matplotlib/seaborn** for visualization
- **scikit-learn** for preprocessing, model training, and evaluation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")


## 2. Load the Titanic Dataset

The notebook expects a file named **`train.csv`**, which is the training dataset from Kaggle's **Titanic - Machine Learning from Disaster** competition.

If you are using Google Colab, the cell will allow you to upload the file if it is not found automatically.


In [ ]:
dataset_path = Path("train.csv")

if not dataset_path.exists():
    try:
        from google.colab import files
        print("Upload the Titanic train.csv file.")
        uploaded = files.upload()
        if uploaded:
            dataset_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        pass

if not dataset_path.exists():
    raise FileNotFoundError(
        "Could not find train.csv. Place train.csv beside this notebook "
        "or upload it in Google Colab."
    )

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 3. Explore the Dataset

We inspect:
- Number of rows and columns
- Column names
- Data types
- Missing values
- Duplicate rows
- Summary statistics
- Survival distribution

This gives us an understanding of the data before preprocessing.


In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all").T)

print("\nSurvival counts:")
display(df["Survived"].value_counts().sort_index().to_frame("Count"))


In [ ]:
# Visualize survival distribution

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Survived")
plt.title("Titanic Survival Distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()


## 4. Select Features and Target

The target variable is **Survived**.

For this case study, we use a practical subset of passenger information:

- `Pclass` – passenger class
- `Sex` – passenger sex
- `Age` – passenger age
- `SibSp` – siblings/spouses aboard
- `Parch` – parents/children aboard
- `Fare` – passenger fare
- `Embarked` – port of embarkation

We intentionally exclude `PassengerId`, `Name`, `Ticket`, and `Cabin` from this beginner-friendly model. Some of these are identifiers or high-cardinality/text fields, and `Cabin` has substantial missingness.


In [ ]:
target = "Survived"

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

missing_columns = [col for col in features + [target] if col not in df.columns]

if missing_columns:
    raise ValueError(
        f"These expected columns are missing: {missing_columns}"
    )

X = df[features].copy()
y = df[target].copy()

print("Features:")
print(features)

print("\nTarget:")
print(target)

display(X.head())


## 5. Split the Dataset

We split the data into:
- **80% training data**
- **20% testing data**

`stratify=y` keeps the proportion of survivors and non-survivors approximately similar in both datasets.

The split is performed before fitting preprocessing steps to avoid data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("Proportion"))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("Proportion"))


## 6. Data Cleaning and Preprocessing

Naive Bayes requires numerical inputs.

### Numerical features
For `Age` and `Fare`, missing values are replaced with the **median** and values are standardized.

### Categorical features
For `Sex` and `Embarked`, missing values are replaced with the most frequent category and categorical values are converted into numerical columns using **One-Hot Encoding**.

The preprocessing is placed inside a scikit-learn pipeline so that the same transformations are applied consistently to training and testing data.


In [ ]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")


## 7. Build and Train the Naive Bayes Model

We use **Gaussian Naive Bayes (`GaussianNB`)**.

The Naive Bayes algorithm estimates the probability of each class using the input features and assumes conditional independence between features given the class.

The complete pipeline performs preprocessing first and then trains the classifier.


In [ ]:
# Naive Bayes pipeline
nb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

# Train the model
nb_model.fit(X_train, y_train)

print("Naive Bayes model trained successfully.")


## 8. Make Predictions

The trained model predicts whether each passenger in the test set survived.

- `0` = Did not survive
- `1` = Survived


In [ ]:
y_pred = nb_model.predict(X_test)

print("First 20 actual values:")
print(y_test.iloc[:20].to_numpy())

print("\nFirst 20 predicted values:")
print(y_pred[:20])


## 9. Evaluate the Model

We calculate:
- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **Confusion Matrix**
- **Classification Report**

These metrics give a more complete view of model performance than accuracy alone.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Model Evaluation")
print("-" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Did Not Survive", "Survived"],
    zero_division=0
))


## 10. Confusion Matrix

A confusion matrix shows:

- **True Negative (TN):** correctly predicted did not survive
- **False Positive (FP):** predicted survived but did not
- **False Negative (FN):** predicted did not survive but survived
- **True Positive (TP):** correctly predicted survived


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Did Not Survive", "Survived"],
    yticklabels=["Did Not Survive", "Survived"]
)

plt.title("Naive Bayes Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)


## 11. Survival Rate by Sex

This exploratory visualization helps us understand one important relationship in the dataset. It is not part of the Naive Bayes algorithm itself, but it helps interpret the data.


In [ ]:
survival_by_sex = (
    df.groupby("Sex")["Survived"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(7, 4))
sns.barplot(
    data=survival_by_sex,
    x="Sex",
    y="Survived"
)

plt.title("Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

display(survival_by_sex)


## 12. Survival Rate by Passenger Class

Passenger class is another useful exploratory variable. The chart shows the proportion of passengers who survived within each class.


In [ ]:
survival_by_class = (
    df.groupby("Pclass")["Survived"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(7, 4))
sns.barplot(
    data=survival_by_class,
    x="Pclass",
    y="Survived"
)

plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

display(survival_by_class)


## 13. Brief Analysis of Results

After running the notebook, replace the placeholders below with your actual values.

### Suggested analysis

> The Naive Bayes classifier was trained to predict passenger survival using passenger class, sex, age, family-related variables, fare, and embarkation port. Missing numerical values were handled using median imputation, while categorical values were imputed and one-hot encoded.
>
> The model achieved a test accuracy of **[insert accuracy]**. Its precision was **[insert precision]**, recall was **[insert recall]**, and F1-score was **[insert F1-score]**.
>
> The confusion matrix shows the number of passengers correctly and incorrectly classified. The exploratory analysis also shows that survival rates varied substantially across passenger groups, particularly by sex and passenger class.
>
> Overall, the model demonstrates how Naive Bayes can be used for a binary classification problem. However, its performance depends on the selected features, preprocessing, dataset size, and assumptions of the algorithm. The results should therefore be interpreted as an educational machine-learning exercise rather than a definitive historical conclusion.


## 14. Final Conclusion

### Key learning points

1. Titanic data contains both numerical and categorical features.
2. Missing values need to be handled before model training.
3. Categorical variables can be converted into numerical features using one-hot encoding.
4. Naive Bayes is a probabilistic classification algorithm.
5. Accuracy alone is not enough; confusion matrix, precision, recall, and F1-score provide additional information.
6. A preprocessing pipeline helps prevent data leakage and makes the workflow reproducible.

### Submission Checklist

- [ ] `KNN_Diabetes.ipynb` completed from Module 21
- [ ] `Naive_Bayes_Titanic.ipynb` completed
- [ ] Titanic `train.csv` used
- [ ] All cells executed successfully
- [ ] Actual model results included in the analysis
- [ ] Notebook saved as `.ipynb`
- [ ] Notebook uploaded to Google Colab
- [ ] Colab sharing set to **Anyone with the link → Viewer**
- [ ] Actual Colab link copied into the submission form
